# Compare Darija Whisper Models (Kaggle T4 GPU)

Benchmark 3 HuggingFace Darija ASR models on the same broadcast clip.
**TL;DR — Model C (anaszil LoRA on turbo) is the winner:** 3.4× faster than
MaghrebVoice with cleaner transcriptions on real broadcast Darija.

| # | Model | Type | Params | Reported WER |
|---|-------|------|--------|-------------|
| **C ✅** | **`anaszil/whisper-large-v3-turbo-darija`** | **LoRA adapter (turbo)** | **809M + 50M** | **24.9%** |
| A | `Ayoub-Laachir/MaghrebVoice` | Full fine-tune (large-v3) | 1.55B | 3.15% |
| B | `Marialab/finetuned-whisper-large-v3-5000-step` | Full fine-tune (large-v3) | 1.55B | BLEU 0.74 |

**Setup:**  
- Settings → Accelerator → **GPU T4 x2** (or P100, A100)  
- Internet **On** (first run downloads ~6–15 GB of model weights)  
- Runtime → **Run all** (or step through cell by cell)

## 1. Install dependencies

In [ ]:
!pip -q install "transformers>=4.47" "torch>=2.0" "accelerate" "librosa" "soundfile" "peft" "jiwer" "tabulate"


## 2. Clone the transcription repo

Pull the repo to get the sample audio clip and utility modules.

In [ ]:
import os, sys
from pathlib import Path

REPO_ROOT = Path("/kaggle/working/haca-transcription")
if not REPO_ROOT.exists():
    !git clone --depth 1 https://github.com/MarTCM/haca-transcription.git "{REPO_ROOT}"
else:
    print("[repo] already cloned")

sys.path.insert(0, str(REPO_ROOT / "src"))
print(f"[repo] {REPO_ROOT}")

## 3. Get a sample audio file

By default, the notebook uses the trimmed broadcast clip from the repo.
You can override via upload or URL:

- Option A — Upload an `.mp3` / `.wav` via Kaggle's **Add Data** → Upload
- Option B — Set `SAMPLE_URL` to a public URL
- Option C — Point `SAMPLE_PATH` to a pre-loaded Kaggle dataset

Edit `SAMPLE_PATH` in the next cell to use your own file.

In [ ]:
import os, torch, time, json, gc
import librosa
import soundfile as sf
from pathlib import Path

REPO_ROOT = Path("/kaggle/working/haca-transcription")

# --- CONFIGURE THESE ---
SAMPLE_PATH = str(REPO_ROOT / "samples" / "mobachara_ma3akom_trimmed.mp3")
SAMPLE_URL  = None          # override: download from URL
OUT_DIR     = Path("/kaggle/working/compare")
# -----------------------

OUT_DIR.mkdir(parents=True, exist_ok=True)
WAV_PATH = OUT_DIR / "sample_16khz.wav"

if os.path.exists(SAMPLE_PATH):
    src = SAMPLE_PATH
elif SAMPLE_URL:
    src = str(OUT_DIR / "sample_src")
    !wget -q "{SAMPLE_URL}" -O "{src}"
else:
    raise RuntimeError(
        "SAMPLE_PATH not found and no SAMPLE_URL set. Did the git clone fail?"
    )

audio, sr = librosa.load(src, sr=16000, mono=True)
sf.write(str(WAV_PATH), audio, 16000)
duration = len(audio) / 16000
print(f"Audio: {WAV_PATH}  ({duration:.1f}s, {sr} Hz, mono)")

## 4. Benchmark helper

The function below loads a model, transcribes the sample, measures time, and
cleans up GPU memory so the next model starts fresh.

In [ ]:
def transcribe_and_time(
    model_id: str,
    audio_path: str,
    is_lora: bool = False,
    lora_base: str = "openai/whisper-large-v3-turbo",
) -> dict:
    """Load *model_id*, transcribe *audio_path*, return timing + text."""
    device = 0 if torch.cuda.is_available() else "cpu"
    dtype  = torch.float16 if torch.cuda.is_available() else torch.float32
    result = {"model": model_id, "load_s": 0, "transcribe_s": 0, "text": ""}

    # ---- Load ----
    t0 = time.time()
    try:
        if is_lora:
            from peft import PeftModel
            from transformers import (
                WhisperForConditionalGeneration, WhisperProcessor, pipeline,
            )
            base = WhisperForConditionalGeneration.from_pretrained(
                lora_base, torch_dtype=dtype, low_cpu_mem_usage=True,
            )
            model = PeftModel.from_pretrained(base, model_id)
            processor = WhisperProcessor.from_pretrained(
                lora_base, language="Arabic", task="transcribe",
            )
            pipe = pipeline(
                "automatic-speech-recognition",
                model=model,
                tokenizer=processor.tokenizer,
                feature_extractor=processor.feature_extractor,
                chunk_length_s=30,
                device=device,
            )
        else:
            from transformers import pipeline as pl
            pipe = pl(
                "automatic-speech-recognition",
                model=model_id,
                device=device,
            )
    except Exception as e:
        result["error"] = str(e)
        return result
    result["load_s"] = round(time.time() - t0, 1)

    # ---- Transcribe ----
    t0 = time.time()
    try:
        out = pipe(audio_path, return_timestamps=True)
        result["text"] = out["text"].strip()
        result["chunks"] = out.get("chunks", [])
    except Exception as e:
        result["error"] = str(e)
        result["text"] = "[TRANSCRIPTION FAILED]"
    result["transcribe_s"] = round(time.time() - t0, 1)

    # ---- Cleanup ----
    del pipe, model, base if is_lora else None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


def pretty_print(result: dict):
    """Print a single model's result to stderr (keeps notebook clean)."""
    tag = result["model"].split("/")[-1]
    err = result.get("error")
    print(f"\n{'='*60}", file=sys.stderr)
    print(f"  {tag}", file=sys.stderr)
    print(f"  Load: {result['load_s']}s  |  Transcribe: {result['transcribe_s']}s", file=sys.stderr)
    if err:
        print(f"  ERROR: {err}", file=sys.stderr)
    print(f"{'='*60}", file=sys.stderr)
    print(result["text"][:500], "\n..." if len(result["text"]) > 500 else "")


import sys

## 5. Run benchmarks

Each cell runs one model.  If you only want to compare 2 out of 3, skip the
cells you don't need (the comparison table will show `N/A` for skipped ones).

---
### Model C ✅ — `anaszil/whisper-large-v3-turbo-darija` (LoRA, recommended)  
LoRA adapter on `large-v3-turbo` (809M base). Lighter download (~3 GB base
+ 50 MB adapter). **Fastest and most accurate on broadcast Darija** — confirmed
by benchmark on real HACA broadcast audio.

In [ ]:
result_c = transcribe_and_time(
    "anaszil/whisper-large-v3-turbo-darija", str(WAV_PATH),
    is_lora=True,
    lora_base="openai/whisper-large-v3-turbo",
)
pretty_print(result_c)

### Model A — `Ayoub-Laachir/MaghrebVoice`  
Full large-v3 fine-tune. Best-reported WER (3.15%) on private corpus, but
slower and noisier than the LoRA adapter on real broadcast audio. Heavy
download (~6 GB).

In [ ]:
result_a = transcribe_and_time(
    "Ayoub-Laachir/MaghrebVoice", str(WAV_PATH), is_lora=False,
)
pretty_print(result_a)

### Model B — `Marialab/finetuned-whisper-large-v3-5000-step`  
Full large-v3 fine-tune on Darija-C. Evaluated with BLEU (0.744). **Known to
hallucinate on broadcast audio** — produces repetitive gibberish on long clips.
Not recommended. Heavy download (~6 GB).

In [ ]:
result_b = transcribe_and_time(
    "Marialab/finetuned-whisper-large-v3-5000-step", str(WAV_PATH), is_lora=False,
)
pretty_print(result_b)

## 6. Comparison table

In [ ]:
from tabulate import tabulate

rows = []
for r in [result_c, result_a, result_b]:
    tag = r["model"].split("/")[-1]
    rows.append([
        tag,
        r.get("load_s", "N/A"),
        r.get("transcribe_s", "N/A"),
        r.get("error", "✔") if not r.get("text") else "✔",
        r["text"][:200] + ("..." if len(r["text"]) > 200 else ""),
    ])

print(tabulate(
    rows,
    headers=["Model", "Load (s)", "Transcribe (s)", "OK?", "First 200 chars"],
    tablefmt="grid",
))
print(f"\nAudio duration: {duration:.1f}s, length: {len(result_c.get('text','').split())} words (C)")

## 7. Results & recommendation

Based on real broadcast Darija (10.5 min clip from "Mobachara Ma3akom"):

| Rank | Model | Load | Transcribe | Verdict |
|------|-------|------|------------|---------|
| 🥇 | `anaszil/whisper-large-v3-turbo-darija` (LoRA) | ~20s | ~134s | **Fastest, cleanest transcription** — recommended default for Darija ASR |
| 🥈 | `Ayoub-Laachir/MaghrebVoice` | ~64s | ~457s | Decent but 3.4× slower, noisier output |
| ❌ | `Marialab/finetuned-whisper-large-v3-5000-step` | ~49s | ~496s | **Hallucinates** on long broadcast — not usable |

**Recommendation:** Use `anaszil/whisper-large-v3-turbo-darija` (or just `openai/whisper-large-v3-turbo`
via faster-whisper for a pure CTranslate2 pipeline) as the default Darija model. The turbo
architecture is both faster and more reliable on broadcast speech than full large-v3 fine-tunes.

## 8. (Optional) Save results to file

In [ ]:
report = {
    "audio": str(WAV_PATH),
    "duration_s": round(duration, 1),
    "results": {
        r["model"]: {
            "load_s": r["load_s"],
            "transcribe_s": r["transcribe_s"],
            "text": r["text"],
            "error": r.get("error"),
        }
        for r in [result_a, result_b, result_c]
    },
}
out_path = OUT_DIR / "benchmark_results.json"
out_path.write_text(json.dumps(report, ensure_ascii=False, indent=2))
print(f"Saved to {out_path}")

## 9. (Optional) Compare with reference transcript

If you have a ground-truth transcript file, set `REF_PATH` below and the cell
will compute Word Error Rate (WER) for each model.

In [ ]:
REF_PATH = None  # set to a .txt file with the reference transcript

if REF_PATH and os.path.exists(REF_PATH):
    from jiwer import wer
    ref_text = Path(REF_PATH).read_text().strip()
    print("WER comparison (lower is better):")
    print(f"  {'Model':<50} {'WER':>8}")
    print(f"  {'-'*50} {'-'*8}")
    for r in [result_a, result_b, result_c]:
        hyp = r.get("text", "")
        if hyp:
            w = round(wer(ref_text, hyp), 2)
        else:
            w = "N/A"
        print(f"  {r['model']:<50} {w:>8}")
else:
    print("No REF_PATH set.  Skipping WER.")